<a href="https://colab.research.google.com/github/Priyall33/Pcos-Endometrosis-risk-model/blob/main/01_setup_and_nhanes_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Notebook 1 - NHANES Data download and Stacking

The goal is to download hormone and repordutive heatlth data from 4 NHANES cycle [2011 - 2020] and combine them into one dataset for modeling

In [5]:
import pandas as pd
import urllib.request #helps download file from the internet
import os #helps interact with computer's file system to check files/create folders

In [4]:
#1. Define all cycles

#each cycle is a 2-3 year period
# TST = hormone blood test result
# RHQ = reproductive health questionnaire answers
cycles = [
    {
        "cycle": "2017-2020",
        "tst_url": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_TST.XPT", #URL for hormone data
        "rhq_url": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_RHQ.XPT", #URL for reporductive health data
        "tst_file": "P_TST.XPT", #name after downloading
        "rhq_file": "P_RHQ.XPT", #name after downloading
    },
    {
        "cycle": "2015-2016",
        "tst_url": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/TST_I.XPT",
        "rhq_url": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/RHQ_I.XPT",
        "tst_file": "TST_I.XPT",
        "rhq_file": "RHQ_I.XPT",
    },
    {
        "cycle": "2013-2014",
        "tst_url": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/TST_H.XPT",
        "rhq_url": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/RHQ_H.XPT",
        "tst_file": "TST_H.XPT",
        "rhq_file": "RHQ_H.XPT",
    },
    {
        "cycle": "2011-2012",
        "tst_url": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/TST_G.XPT",
        "rhq_url": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/RHQ_G.XPT",
        "tst_file": "TST_G.XPT",
        "rhq_file": "RHQ_G.XPT",
    },
]

#  2. Download all files
print("=== Downloading NHANES files ===")

#loop through each cycle one at a time
for c in cycles:

#loop through both the hormone file and questionnaire file
    for key in ["tst", "rhq"]:

      #get URL and Filename for this sepecific file
        url  = c[f"{key}_url"]
        path = c[f"{key}_file"]

        #checking if file is dowloaded before
        if os.path.exists(path):
            print(f"  Already exists: {path}")
        else:
          #download if doesn't exist
            print(f"  Downloading {path} ...")

          #save to computer
            urllib.request.urlretrieve(url, path)
            print(f"  ✅ Saved: {path}")

# 3. Hormone columns present across all cycles
# Each cycle may differ slightly; we keep the intersection of key biomarkers

# NHANES uses coded column names so here's what each one means:
# SEQN    = participant ID number (unique for every person)
# LBXAMH  = Anti-Mullerian Hormone - elevated in PCOS
# LBXFSH  = Follicle Stimulating Hormone - indicates ovarian function
# LBXLUH  = Luteinizing Hormone - triggers ovulation
# LBXSHBG = Sex Hormone Binding Globulin - low = more free testosterone
# LBXAND  = Androstenedione - elevated in PCOS/hyperandrogenism
# LBXEST  = Estradiol - elevated in endometriosis
# LBXPG4  = Progesterone - tracks menstrual cycle phase

HORMONE_COLS = ["SEQN", "LBXAMH", "LBXFSH", "LBXLUH", "LBXSHBG",
                "LBXAND", "LBXEST", "LBXPG4"]


# RHQ305  = "Were you ever told by a doctor you had endometriosis?"  1 = Yes, 2 = No (this is our TARGET LABEL)
# RHQ010  = Age at first menstrual period
# RHD043  = Age at last menstrual period
# RHQ131  = Ever been pregnant
# RHQ160  = Ever told you had uterine fibroids
# RHQ162  = Age when told about fibroids

RHQ_COLS     = ["SEQN", "RHQ305", "RHQ010", "RHD043", "RHQ131",
                "RHQ160", "RHQ162"]

# 4. Load, merge, and stack each cycle
print("\n=== Loading and merging cycles ===")

#empty list will hold data from each cycle
all_cycles = []

for c in cycles:
    cycle_name = c["cycle"]
    #loads hormone blood in tbale "tst" and reproducive health in "rhq"
    tst = pd.read_sas(c["tst_file"], format="xport", encoding="utf-8")
    rhq = pd.read_sas(c["rhq_file"], format="xport", encoding="utf-8")

    # Keep only columns that exist in this cycle
    tst_keep = [col for col in HORMONE_COLS if col in tst.columns]
    rhq_keep = [col for col in RHQ_COLS     if col in rhq.columns]

    #combining hormone data with diagnosis data
    #pd.merge finds rows where SEQN matches in both and combines them
    # "inner" keeps people when they inclde both or they get dropped
    merged = pd.merge(tst[tst_keep], rhq[rhq_keep], on="SEQN", how="inner")
    merged["cycle"] = cycle_name  # tag which cycle each row came from

    # Filter to females with a valid endometriosis label only any other gets dropped
    if "RHQ305" in merged.columns:
        merged = merged[merged["RHQ305"].isin([1.0, 2.0])]

    endo_pos = (merged["RHQ305"] == 1.0).sum() if "RHQ305" in merged.columns else 0
    print(f"  {cycle_name}: {len(merged):,} rows | {endo_pos} endometriosis positive")
    all_cycles.append(merged),

# 5. Stack all cycles in one dataset
df = pd.concat(all_cycles, ignore_index=True)

print(f"\n=== Final stacked dataset ===")
print(f"  Total rows      : {len(df):,}")
print(f"  Endo positive   : {(df['RHQ305'] == 1.0).sum():,}")
print(f"  Endo negative   : {(df['RHQ305'] == 2.0).sum():,}")
print(f"  Columns         : {list(df.columns)}")

# 6. Create binary label column (target)
df["endometriosis"] = (df["RHQ305"] == 1.0).astype(int)

# 7. Save to CSV
output_file = "nhanes_stacked_all_cycles.csv"
df.to_csv(output_file, index=False)
print(f"\n✅ Saved stacked dataset to: {output_file}")
print(f"   Shape: {df.shape}")

=== Downloading NHANES files ===
  Already exists: P_TST.XPT
  Already exists: P_RHQ.XPT
  Already exists: TST_I.XPT
  Already exists: RHQ_I.XPT
  Already exists: TST_H.XPT
  Already exists: RHQ_H.XPT
  Already exists: TST_G.XPT
  Already exists: RHQ_G.XPT

=== Loading and merging cycles ===
  2017-2020: 3,985 rows | 471 endometriosis positive
  2015-2016: 2,505 rows | 287 endometriosis positive
  2013-2014: 2,601 rows | 304 endometriosis positive
  2011-2012: 2,253 rows | 266 endometriosis positive

=== Final stacked dataset ===
  Total rows      : 11,344
  Endo positive   : 1,328
  Endo negative   : 10,016
  Columns         : ['SEQN', 'LBXAMH', 'LBXFSH', 'LBXLUH', 'LBXSHBG', 'LBXAND', 'LBXEST', 'LBXPG4', 'RHQ305', 'RHQ010', 'RHD043', 'RHQ131', 'RHQ160', 'RHQ162', 'cycle']

✅ Saved stacked dataset to: nhanes_stacked_all_cycles.csv
   Shape: (11344, 16)


In [ ]:
from google.colab import files
files.download('nhanes_stacked_all_cycles.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>